# Business Intelligence in Action — Data Analytics Examination
**Unit 12: Data Analytics | BTEC Level 4 | Unit Code: K/615/1637**

| | |
|---|---|
| **Dataset** | Brazilian E-Commerce Public Dataset (Olist) |
| **Source** | kaggle.com/datasets/olistbr/brazilian-ecommerce |
| **Tool** | Python 3 · pandas · numpy · matplotlib · seaborn · scikit-learn |
| **Assessor** | Nilufar Nosirjonova |


---
# TASK 1 — Theoretical Foundation of Data Analytics
### Criteria: P1 · P2 · M1 · D1
---
## 1.1 Dataset Introduction (P1)

The **Olist Brazilian E-Commerce Dataset** contains 112,650 order-item records from an online marketplace in Brazil (October 2016 – August 2018). It includes product price, freight cost, product category, customer location, and order date.

**Business problem:** Olist wants to understand what drives revenue, which product categories perform best, and forecast future sales to make better business decisions.

**What is Data Analytics?**
Data analytics is the process of examining raw data to discover useful patterns and support better business decisions (Provost & Fawcett, 2013).

| Term | Definition | Olist Example |
|------|-----------|---------------|
| **Population** | All records in the full dataset | All 112,650 order-item records |
| **Sample** | A smaller representative subset | Random 10% = ~11,265 records |
| **Nominal data** | Categories with no order | `customer_state` (SP, RJ, MG…) |
| **Ordinal data** | Categories with a meaningful order | `review_score` (1 = worst, 5 = best) |
| **Continuous data** | Any decimal number | `price` (R$74.99), `freight_value` |
| **Discrete data** | Whole countable numbers only | `order_item_id` (1, 2, 3…) |
| **Descriptive analytics** | Summarises what already happened | Mean price = R$120.65 |
| **Predictive analytics** | Forecasts what will happen | Regression predicting revenue from price |

---
## 1.2 Load Libraries and Dataset (P2)


In [ ]:
# CODE 1.1 — Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# CODE 1.2 — Load the 8 CSV files
DATA = 'ecommerce_data/'   # your folder path

orders      = pd.read_csv(DATA + 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp'])
order_items = pd.read_csv(DATA + 'olist_order_items_dataset.csv')
products    = pd.read_csv(DATA + 'olist_products_dataset.csv')
customers   = pd.read_csv(DATA + 'olist_customers_dataset.csv')
cat_trans   = pd.read_csv(DATA + 'product_category_name_translation.csv')

print(f"orders:      {orders.shape}")
print(f"order_items: {order_items.shape}")
print(f"products:    {products.shape}")

In [ ]:
# CODE 1.3 — Merge all tables into one main DataFrame
products_en = products.merge(cat_trans, on='product_category_name', how='left')

main = (order_items
        .merge(orders[['order_id','customer_id','order_purchase_timestamp']], on='order_id')
        .merge(products_en[['product_id','product_category_name_english']], on='product_id', how='left')
        .merge(customers[['customer_id','customer_state']], on='customer_id', how='left'))

main['revenue']    = main['price'] + main['freight_value']
main['year_month'] = main['order_purchase_timestamp'].dt.to_period('M')

print(f"Main table: {main.shape[0]:,} rows x {main.shape[1]} columns")

In [ ]:
# CODE 1.4 — Preview the dataset (P2 screenshot evidence)
display(main.head())
main.info()

**P2 Evidence:** The output above shows all column names, data types, and row count (112,650) — confirming the dataset loaded successfully into Python.

---
## 1.3 Three Types of Analytics (M1)

### Descriptive Analytics — *"What happened?"*
Summarises historical data. **Industry example:** Amazon uses descriptive dashboards to monitor daily sales by category and region — letting managers spot underperforming products within hours.
**In Olist:** São Paulo (SP) = 42% of all orders. Top revenue category = `health_beauty`.

### Predictive Analytics — *"What will happen?"*
Forecasts future outcomes using past data. **Industry example:** Mercado Livre uses demand forecasting to pre-position inventory at regional warehouses before seasonal peaks.
**In Olist:** Linear regression predicts revenue from price (R² = 0.9885). Time series forecasts monthly revenue for next 3 months.

### Prescriptive Analytics — *"What should we do?"*
Recommends the best action based on predictions. **Industry example:** DHL uses route optimisation to find the cheapest delivery route in real time.
**In Olist:** Based on forecasts, recommend increasing inventory and seller recruitment for `health_beauty` before the November peak season.

---
## 1.4 Critical Evaluation (D1)

Data analytics transforms raw transaction records into actionable intelligence. Without it, the Olist client cannot identify which categories drive revenue, which regions are growing fastest, or what will happen next quarter.

The descriptive analysis in Task 2 showed that the standard deviation of product prices is R$183.63 — higher than the mean of R$120.65. This extreme variability means a single "average price" is useless for strategy. A business cannot set promotions, manage inventory, or plan logistics without knowing this — and it is invisible without analytics applied to all 112,650 records.

The regression in Task 3 quantified that every R$1 increase in price adds R$1.05 in revenue (R² = 0.9885), giving the client a clear pricing lever. This is what James et al. (2021) call the core value of statistical learning: uncovering high-impact patterns at scale.

However, analytics has limits. Correlation does not mean causation. The dataset ends August 2018, so long-term forecasts carry uncertainty. Models built on historical data can fail when market conditions change (Bruce & Bruce, 2020). Analytics should support human judgment, not replace it.

---


---
# TASK 2 — Descriptive Analytics
### Criteria: P3 · P4 · M2
---
## 2.1 Missing Values (P3)


In [ ]:
# CODE 2.1 — Find missing values
missing = main.isnull().sum()
missing_pct = (missing / len(main) * 100).round(2)
print(missing[missing > 0])
print(missing_pct[missing_pct > 0])

In [ ]:
# CODE 2.2 — Fix missing values
main['product_category_name_english'] = main['product_category_name_english'].fillna('Unknown')
main['freight_value'] = main['freight_value'].fillna(main['freight_value'].median())

print(f"Remaining nulls: {main.isnull().sum().sum()}")

**Explanation (P3):**
- `product_category_name_english`: 1,627 missing (1.44%) — filled with 'Unknown' to keep all rows.
- `freight_value`: filled with median — median is more robust than mean for right-skewed data (Bruce & Bruce, 2020).


---
## 2.2 Univariate Analysis — Price Distribution (P3)

In [ ]:
# CODE 2.3 — Figure 1: Histogram of product prices
main['price'].clip(upper=500).hist(bins=40, color='steelblue', edgecolor='white')
plt.axvline(main['price'].mean(),   color='red',   lw=2, ls='--', label=f"Mean: R${main['price'].mean():.2f}")
plt.axvline(main['price'].median(), color='green', lw=2, ls='--', label=f"Median: R${main['price'].median():.2f}")
plt.title('Figure 1: Product Price Distribution')
plt.xlabel('Price (R$)'); plt.ylabel('Count')
plt.legend(); plt.tight_layout(); plt.show()

**Figure 1 Interpretation (P3):** A histogram was chosen to show the shape of one continuous variable. Strong right skew — most products cost below R$200, but a few expensive items pull the mean (R$120.65) above the median (R$74.99). The median is the better "typical" price.


### Univariate Analysis — Top Categories by Revenue (P3)

In [ ]:
# CODE 2.4 — Figure 2: Top 10 categories by revenue
cat_rev = main.groupby('product_category_name_english')['revenue'].sum().sort_values(ascending=False).head(10)
cat_rev[::-1].plot(kind='barh', color='steelblue')
plt.title('Figure 2: Top 10 Product Categories by Revenue')
plt.xlabel('Total Revenue (R$)'); plt.tight_layout(); plt.show()

**Figure 2 Interpretation (P3):** A horizontal bar chart was chosen because category names are long. `health_beauty` is the highest revenue category. The top 3 categories generate a disproportionately large share of total revenue — creating concentration risk if any one is disrupted.


---
## 2.3 Bivariate Analysis — Price vs Freight (P3)

In [ ]:
# CODE 2.5 — Figure 3: Scatter plot — Price vs Freight Value
sample_sc = main[['price','freight_value']].sample(3000, random_state=42)
r_val, _ = stats.pearsonr(sample_sc['price'], sample_sc['freight_value'])

plt.scatter(sample_sc['price'], sample_sc['freight_value'], alpha=0.3, s=12, color='steelblue')
plt.xlim(0, 500); plt.ylim(0, 80)
plt.xlabel('Price (R$)'); plt.ylabel('Freight Value (R$)')
plt.title(f'Figure 3: Price vs Freight  (Pearson r = {r_val:.3f})')
plt.tight_layout(); plt.show()

print(f"Pearson r = {r_val:.4f}")

**Figure 3 Interpretation (P3):** A scatter plot was chosen to show the relationship between two continuous variables. Pearson r = 0.268 — weak positive correlation. Higher-priced items tend to cost slightly more to ship, but freight is mainly driven by product size/weight, not price alone.


---
## 2.4 Descriptive Statistics (P3 · P4)
### Central Tendency

In [ ]:
# CODE 2.6 — Mean, Median, Mode for price, freight, and revenue
for col in ['price', 'freight_value', 'revenue']:
    print(f"{col:<15}  Mean=R${main[col].mean():.2f}  Median=R${main[col].median():.2f}  Mode=R${main[col].mode()[0]:.2f}")

**Interpretation (P3):** For all three variables: Mean > Median > Mode — the classic signature of right skew. The mean is pulled up by expensive premium items. The **median** is the better measure of a typical transaction for the client's financial reporting.


### Dispersion

In [ ]:
# CODE 2.7 — Range, Variance, Standard Deviation
for col in ['price', 'freight_value', 'revenue']:
    print(f"{col:<15}  Range=R${main[col].max()-main[col].min():.2f}  Variance={main[col].var():,.0f}  StdDev=R${main[col].std():.2f}")

**Interpretation (P3):** Price standard deviation (R$183.63) is **higher than the mean** (R$120.65) — extreme variability. The catalogue spans from R$0.85 accessories to R$6,735 luxury goods. One "average price" cannot represent this range meaningfully.


### Position — Quartiles and Outliers

In [ ]:
# CODE 2.8 — Q1, Q3, IQR, outlier count for price
q1  = main['price'].quantile(0.25)
q3  = main['price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = ((main['price'] < lower) | (main['price'] > upper)).sum()

print(f"Q1=R${q1:.2f}  Q3=R${q3:.2f}  IQR=R${iqr:.2f}")
print(f"Lower fence=R${lower:.2f}  Upper fence=R${upper:.2f}")
print(f"Outliers: {outliers:,} ({outliers/len(main)*100:.1f}%)")

**Interpretation (P3):** 8,427 price outliers (7.5%) are above R$277.40. These are premium products and should **not** be deleted — they often represent the highest-margin transactions (Knaflic, 2015).


---
## 2.5 Visualisations (P4)

In [ ]:
# CODE 2.9 — Figure 4: Box plot — Price by top 6 categories
top6 = cat_rev.head(6).index.tolist()
df6  = main[main['product_category_name_english'].isin(top6) & (main['price'] <= 600)]

df6.boxplot(column='price', by='product_category_name_english', figsize=(12,6))
plt.title('Figure 4: Price Distribution by Top 6 Categories')
plt.suptitle(''); plt.xlabel('Category'); plt.ylabel('Price (R$)')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

**Figure 4 Interpretation (P4):** A box plot was chosen to compare spread, median, and outliers across multiple groups at once. `watches_gifts` has the highest median price and widest spread — it mixes luxury watches with cheap gifts. `health_beauty` has a compact, moderate price range consistent with a high-volume mass-market category.


In [ ]:
# CODE 2.10 — Figure 5: Monthly revenue line chart
monthly = main.groupby('year_month')['revenue'].sum().reset_index()
monthly['ym_dt'] = monthly['year_month'].dt.to_timestamp()
monthly = monthly[(monthly['ym_dt'] >= '2017-01-01') & (monthly['ym_dt'] <= '2018-08-31')]

plt.figure(figsize=(12, 5))
plt.plot(monthly['ym_dt'], monthly['revenue']/1e6, color='steelblue', lw=2.5, marker='o', ms=4)
plt.fill_between(monthly['ym_dt'], monthly['revenue']/1e6, alpha=0.15, color='steelblue')
plt.xlabel('Month'); plt.ylabel('Revenue (R$ Millions)')
plt.title('Figure 5: Monthly Revenue Trend 2017–2018')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

**Figure 5 Interpretation (P4):** A line chart was chosen because it shows continuous change in a metric over time. Revenue grew ~6x from January to November 2017, then stabilised around R$1.0–1.1M. The November 2017 peak likely reflects pre-Christmas shopping demand.


---
## 2.6 Sampling — 10% Sample vs Population (P3)

In [ ]:
# CODE 2.11 — 10% random sample vs population comparison
sample = main.sample(frac=0.10, random_state=42)
print(f"Population: {len(main):,}  |  Sample: {len(sample):,}")
print()
for col in ['price', 'freight_value', 'revenue']:
    pop_m = main[col].mean()
    sam_m = sample[col].mean()
    diff  = abs(pop_m - sam_m) / pop_m * 100
    print(f"{col:<15}  Population=R${pop_m:.2f}  Sample=R${sam_m:.2f}  Diff={diff:.3f}%")

**Interpretation (P3):** All differences under 0.7% — the sample is highly representative. With 11,000+ records, the Central Limit Theorem guarantees the sample mean will closely match the population mean (Bruce & Bruce, 2020).


---
## 2.7 Linking Descriptive Analytics to Decision-Making (M2)

**Finding 1 — High Price Variability (StdDev = R$183.63):**
Cannot use one pricing strategy for all products. *Recommendation:* Split the catalogue into Budget (<R$40), Mid-range (R$40–R$135), and Premium (>R$135) tiers with separate promotional strategies for each.

**Finding 2 — Health & Beauty Dominates Revenue:**
Concentration risk if this category declines. *Recommendation:* Invest in recruiting more health & beauty sellers, negotiate better logistics rates for this category, and plan campaigns around Mother's Day and Valentine's Day peaks.

**Finding 3 — São Paulo = 42% of Orders:**
Over-reliance on one state. *Recommendation:* Allocate marketing budget to expand seller coverage in the Northeast (BA, CE) and South (RS, PR) to diversify geographic revenue and reduce regional risk.

---


---
# TASK 3 — Predictive Analytics
### Criteria: P5 · P6 · M3 · D2
---
## 3.1 What is Predictive Analytics? (P5)

| | Descriptive | Predictive |
|---|---|---|
| Question | What happened? | What will happen? |
| Output | Charts, statistics | Forecasts, scores |
| Techniques | Mean, SD, correlation | Regression, time series |
| Business use | Reporting | Planning, budgeting |

Two techniques applied:
1. **Simple Linear Regression** — predict revenue from price
2. **Time Series Forecasting** — Moving Average + Exponential Smoothing + Trend Projection

---
## 3.2 Simple Linear Regression — Price → Revenue (P5 · P6)


In [ ]:
# CODE 3.1 — Simple Linear Regression: predict revenue from price
reg = main[['price','revenue']].dropna()
reg = reg[reg['price'] <= 1000]

X = reg[['price']].values
y = reg['revenue'].values

model1  = LinearRegression().fit(X, y)
y_pred1 = model1.predict(X)
r2_1    = r2_score(y, y_pred1)
rmse_1  = np.sqrt(mean_squared_error(y, y_pred1))

print(f"Equation : Revenue = {model1.intercept_:.2f} + {model1.coef_[0]:.4f} x Price")
print(f"R2       = {r2_1:.4f}  ({r2_1*100:.2f}% variance explained)")
print(f"RMSE     = R${rmse_1:.2f}")
print(f"Meaning  : R$1 rise in price --> revenue rises by R${model1.coef_[0]:.4f}")

In [ ]:
# CODE 3.2 — Figure 6: Regression line over actual data
idx   = np.random.default_rng(42).choice(len(X), 2000, replace=False)
xline = np.linspace(0, 1000, 300).reshape(-1, 1)

plt.scatter(X[idx], y[idx], alpha=0.2, s=10, color='steelblue', label='Actual (n=2000)')
plt.plot(xline, model1.predict(xline), color='red', lw=2.5,
         label=f'Fitted line (R2={r2_1:.4f})')
plt.xlabel('Price (R$)'); plt.ylabel('Revenue (R$)')
plt.title('Figure 6: Simple Linear Regression — Price vs Revenue')
plt.legend(); plt.tight_layout(); plt.show()

**Figure 6 Interpretation (P5/P6):** R² = 0.9885 means price explains 98.85% of revenue variation. The slope of 1.0487 tells us each R$1 increase in price adds R$1.05 in revenue — the slight excess over 1.0 reflects that higher-priced items tend to have marginally higher freight charges.


---
## 3.3 Time Series Forecasting — Monthly Revenue (P5 · P6)

In [ ]:
# CODE 3.3 — Build monthly revenue time series
monthly_ts = main.groupby('year_month')['revenue'].sum().reset_index()
monthly_ts['ym_dt'] = monthly_ts['year_month'].dt.to_timestamp()
monthly_ts = monthly_ts[(monthly_ts['ym_dt'] >= '2017-01-01') &
                        (monthly_ts['ym_dt'] <= '2018-08-31')].reset_index(drop=True)

print(f"Months in series: {len(monthly_ts)}")
print(f"Peak month: {monthly_ts.loc[monthly_ts['revenue'].idxmax(),'ym_dt'].strftime('%B %Y')}")
print(f"Peak value: R${monthly_ts['revenue'].max():,.0f}")

In [ ]:
# CODE 3.4 — Apply three forecasting methods

# Method 1: 3-month Moving Average
monthly_ts['MA3'] = monthly_ts['revenue'].rolling(3).mean()

# Method 2: Exponential Smoothing (alpha=0.3)
alpha = 0.3
es = [monthly_ts['revenue'].iloc[0]]
for v in monthly_ts['revenue'].iloc[1:]:
    es.append(alpha * v + (1 - alpha) * es[-1])
monthly_ts['ES'] = es

# Method 3: Trend Projection (last 6 months → next 3 months)
last6 = monthly_ts.tail(6)
slope, intercept, *_ = stats.linregress(np.arange(6), last6['revenue'])
future_dates  = pd.date_range('2018-09-01', periods=3, freq='MS')
forecast_vals = intercept + slope * np.arange(6, 9)

print("3-Month Forecast (Sep–Nov 2018):")
for dt, fv in zip(future_dates, forecast_vals):
    print(f"  {dt.strftime('%B %Y')}: R${fv:,.0f}")

In [ ]:
# CODE 3.5 — Figure 7: Full forecast chart
plt.figure(figsize=(13, 6))
plt.plot(monthly_ts['ym_dt'], monthly_ts['revenue']/1e6, color='steelblue', lw=2.5, marker='o', ms=4, label='Actual Revenue')
plt.plot(monthly_ts['ym_dt'], monthly_ts['MA3']/1e6, color='orange', lw=2, ls='--', label='Moving Average (3-month)')
plt.plot(monthly_ts['ym_dt'], monthly_ts['ES']/1e6, color='green', lw=1.8, ls=':', label='Exponential Smoothing (a=0.3)')
plt.plot(future_dates, forecast_vals/1e6, color='red', lw=2.5, ls='--', marker='D', ms=8, label='Trend Forecast (Sep–Nov 2018)')
plt.axvline(pd.Timestamp('2018-08-01'), color='grey', lw=1.2, ls=':')
plt.xlabel('Month'); plt.ylabel('Revenue (R$ Millions)')
plt.title('Figure 7: Monthly Revenue — Actual, Smoothed, and Forecast')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45); plt.legend(fontsize=9); plt.tight_layout(); plt.show()

**Figure 7 Interpretation (P5/P6):** The grey vertical line separates historical data from the forecast. Moving Average lags behind trend changes by ~1.5 months. Exponential Smoothing responds faster because it weights recent months more. The red forecast shows a slight declining trend for Sep–Nov 2018 based on the most recent months.


---
## 3.4 Comparing the Two Techniques (M3)

In [ ]:
# CODE 3.6 — Compare Moving Average vs Exponential Smoothing accuracy (MAE)
compare = monthly_ts[['revenue','MA3','ES']].dropna()
mae_ma3 = (compare['revenue'] - compare['MA3']).abs().mean()
mae_es  = (compare['revenue'] - compare['ES']).abs().mean()

print(f"Moving Average (3-month) MAE: R${mae_ma3:,.0f}")
print(f"Exponential Smoothing   MAE: R${mae_es:,.0f}")
print(f"
Better method: {'Exponential Smoothing' if mae_es < mae_ma3 else 'Moving Average'}")

**M3 Comparison:**

| | Simple Linear Regression | Moving Average (MA3) | Exponential Smoothing |
|---|---|---|---|
| **Predicts** | Revenue from price | Next month's revenue | Next month's revenue |
| **R² / MAE** | R² = 0.9885 | Higher MAE | Lower MAE |
| **Assumption** | Linear price–revenue relation | All recent months equal weight | Recent months weighted more |
| **Lag** | None | ~1.5 months | ~0.5 months |
| **Best use** | Pricing strategy decisions | Stable, flat data | Trending or seasonal data |
| **Limitation** | Revenue derived from price | Slow to react to trend changes | Alpha value needs tuning |

**Recommendation:** Use Exponential Smoothing for monthly revenue planning (lower error, faster response). Use Simple Linear Regression for pricing decisions ("each R$1 price increase adds R$1.05 revenue per item").


---
## 3.5 Critical Evaluation (D2)

**Regression confidence (R² = 0.9885):**
This appears excellent but must be interpreted carefully. Revenue is mathematically derived from price and freight in this dataset — so a near-perfect R² is partly structural. In genuine predictive models with truly independent variables, R² of 0.30–0.60 is more typical and considered meaningful (James et al., 2021). The model is best used for "what if" pricing scenarios — e.g. "if we raise average price by R$20, expected revenue rises by R$20.97 per item."

**Time series forecast limitations:**

1. **Data truncation:** The dataset ends August 2018. The final months typically show lower recorded orders in e-commerce data because late deliveries haven't been captured yet — so the apparent July–August decline may be a data artefact, not a real business slowdown.

2. **Short series:** Only 20 months — too short to reliably separate seasonal patterns from underlying trend. James et al. (2021) recommend at least 2–3 full seasonal cycles before applying seasonal models.

3. **No external variables:** Brazil's GDP growth, inflation, competitor activity (Mercado Livre, Amazon Brasil), and Olist's marketing spend are all absent from the model. These real-world forces can easily override any trend projected from revenue history alone.

**Responsible use of forecasts:**
The Sep–Nov 2018 projections should be treated as planning anchors, not financial commitments. McKinney (2022) advises always communicating model outputs with uncertainty bounds. Testing an optimistic scenario (flat trend) and a pessimistic scenario (faster decline) provides a more useful planning range than a single point forecast.

Despite these limits, the predictive analysis adds real strategic value: it shifts the client from purely looking backwards to planning forwards — the defining mark of a genuinely data-driven organisation (Marr, 2017).

---


---
## References (Harvard Style)

Bruce, P. and Bruce, A. (2020) *Practical Statistics for Data Scientists*. 2nd edn. Sebastopol: O'Reilly Media.

Few, S. (2012) *Show Me the Numbers*. 2nd edn. Burlingame: Analytics Press.

James, G., Witten, D., Hastie, T. and Tibshirani, R. (2021) *An Introduction to Statistical Learning*. 2nd edn. New York: Springer.

Knaflic, C. N. (2015) *Storytelling with Data*. USA: John Wiley & Sons.

Marr, B. (2017) *Data Strategy*. London: Kogan Page.

McKinney, W. (2022) *Python for Data Analysis*. 3rd edn. Sebastopol: O'Reilly Media.

Provost, F. and Fawcett, T. (2013) *Data Science for Business*. Sebastopol: O'Reilly Media.
